In [ ]:
# =========================================
# 0) INSTALLS (Colab)
# =========================================
!pip -q install albumentations==1.4.7 pycocotools timm

In [ ]:
# =========================================================
# CELL 2: Imports + Config
# =========================================================
import os, json, random, math
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.ops import box_iou

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

SEED = 42
IMG_SIZE = 512
BATCH_SIZE = 4
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from google.colab import drive
drive.mount('/content/drive')
ROOT = Path("/content/drive/MyDrive/bm616/data")   # <-- change if needed
DATASET_NAMES = ["stenosis", "syntax"]  # train sequentially on both

# epochs
EPOCHS_SEGCLS = 10
EPOCHS_DET = 10

# loss weights for seg+cls
LAMBDA_SEG = 1.0
LAMBDA_CLS = 0.2

print("Device:", DEVICE)

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(SEED)

In [ ]:
# =========================================================
# CELL 3: COCO utilities
# =========================================================
def load_coco(split, dataset_name):
    ann_path = ROOT / dataset_name / split / "annotations" / f"{split}.json"
    img_dir = ROOT / dataset_name / split / "images"
    with open(ann_path, "r") as f:
        coco = json.load(f)

    images = coco.get("images", [])
    annotations = coco.get("annotations", [])
    categories = coco.get("categories", [])

    anns_by_img = defaultdict(list)
    for a in annotations:
        anns_by_img[a["image_id"]].append(a)

    file_lookup = {im["id"]: im["file_name"] for im in images}
    return images, anns_by_img, file_lookup, img_dir, categories

def coco_poly_to_mask(anns, h, w):
    mask = np.zeros((h, w), dtype=np.uint8)
    for ann in anns:
        seg = ann.get("segmentation", [])
        if isinstance(seg, list):
            for poly in seg:
                pts = np.array(poly).reshape(-1, 2).astype(np.int32)
                if len(pts) >= 3:
                    cv2.fillPoly(mask, [pts], 1)
    return mask

def coco_bbox_xywh_to_xyxy(b):
    x, y, w, h = b
    return [x, y, x + w, y + h]

In [ ]:
# =========================================================
# CELL 4: Transforms
# =========================================================
def get_segcls_tfm(train=True):
    if train:
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.CLAHE(p=0.4),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(0.05, 0.1, 15, p=0.5),
            A.GaussNoise(p=0.2),
            A.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5)),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5)),
            ToTensorV2()
        ])

# detection tfm with bbox support
def get_det_tfm(train=True):
    if train:
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(0.05, 0.1, 10, p=0.4),
        ], bbox_params=A.BboxParams(format="pascal_voc", label_fields=["labels"], min_visibility=0.1))
    else:
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
        ], bbox_params=A.BboxParams(format="pascal_voc", label_fields=["labels"], min_visibility=0.1))

In [ ]:
# =========================================================
# CELL 5: Dataset for Seg + Cls
# =========================================================
class SegClsDataset(Dataset):
    def __init__(self, split, dataset_name):
        self.images, self.anns_by_img, self.lookup, self.img_dir, _ = load_coco(split, dataset_name)
        self.tfm = get_segcls_tfm(train=(split=="train"))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        item = self.images[idx]
        img_id = item["id"]

        img_path = self.img_dir / self.lookup[img_id]
        gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            raise FileNotFoundError(str(img_path))

        anns = self.anns_by_img.get(img_id, [])
        mask = coco_poly_to_mask(anns, gray.shape[0], gray.shape[1])
        cls_label = 1.0 if len(anns) > 0 else 0.0

        img3 = np.stack([gray, gray, gray], axis=-1)

        aug = self.tfm(image=img3, mask=mask)
        x = aug["image"].float()
        y_seg = aug["mask"].unsqueeze(0).float()
        y_cls = torch.tensor(cls_label, dtype=torch.float32)

        return x, y_seg, y_cls

In [ ]:
# =========================================================
# CELL 6: Model for Seg + Cls
# =========================================================
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, 2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class SegClsNet(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

        self.stem = nn.Sequential(base.conv1, base.bn1, base.relu)  # /2
        self.pool = base.maxpool                                    # /4
        self.e1 = base.layer1                                       # 256
        self.e2 = base.layer2                                       # 512
        self.e3 = base.layer3                                       # 1024
        self.e4 = base.layer4                                       # 2048

        # cls
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(2048, 1)
        )

        # seg decoder
        self.d3 = DecoderBlock(2048, 1024, 512)
        self.d2 = DecoderBlock(512, 512, 256)
        self.d1 = DecoderBlock(256, 256, 128)
        self.d0 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.seg_out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        s = self.stem(x)
        p = self.pool(s)
        f1 = self.e1(p)
        f2 = self.e2(f1)
        f3 = self.e3(f2)
        f4 = self.e4(f3)

        cls_logit = self.cls_head(f4).squeeze(1)

        z = self.d3(f4, f3)
        z = self.d2(z, f2)
        z = self.d1(z, f1)
        z = self.d0(z)
        seg_logit = self.seg_out(z)
        seg_logit = F.interpolate(seg_logit, size=x.shape[-2:], mode="bilinear", align_corners=False)

        return cls_logit, seg_logit

def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    inter = (probs * targets).sum(dim=(1,2,3))
    union = probs.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3))
    dice = (2*inter + eps)/(union + eps)
    return 1.0 - dice.mean()

def segcls_total_loss(cls_logit, seg_logit, y_cls, y_seg, bce_cls, bce_seg):
    l_cls = bce_cls(cls_logit, y_cls)
    l_seg = 0.5*bce_seg(seg_logit, y_seg) + 0.5*dice_loss(seg_logit, y_seg)
    total = LAMBDA_CLS*l_cls + LAMBDA_SEG*l_seg
    return total, l_cls.item(), l_seg.item()

In [ ]:
# =========================================================
# CELL 7: Train/Val functions for Seg+Cls
# =========================================================
def run_segcls_epoch(model, loader, optimizer, bce_cls, bce_seg, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total, cls_l, seg_l = 0.0, 0.0, 0.0
    preds_all, tgts_all = [], []

    for i, (x, y_seg, y_cls) in enumerate(loader):
        x = x.to(DEVICE, non_blocking=True)
        y_seg = y_seg.to(DEVICE, non_blocking=True)
        y_cls = y_cls.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            cls_logit, seg_logit = model(x)
            loss, lcls, lseg = segcls_total_loss(cls_logit, seg_logit, y_cls, y_seg, bce_cls, bce_seg)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total += loss.item()
        cls_l += lcls
        seg_l += lseg

        p = (torch.sigmoid(cls_logit) > 0.5).int().detach().cpu().numpy()
        t = y_cls.int().detach().cpu().numpy()
        preds_all.extend(p)
        tgts_all.extend(t)

        if train and i % 25 == 0:
            print(f"[train] batch {i}/{len(loader)} loss={loss.item():.4f} cls={lcls:.4f} seg={lseg:.4f}", flush=True)

    preds_all = np.array(preds_all)
    tgts_all = np.array(tgts_all)
    acc = (preds_all == tgts_all).mean() if len(tgts_all) else 0.0
    cm = confusion_matrix(tgts_all, preds_all, labels=[0,1])
    p, r, f1, _ = precision_recall_fscore_support(tgts_all, preds_all, average="binary", zero_division=0)

    n = max(1, len(loader))
    return total/n, cls_l/n, seg_l/n, acc, p, r, f1, cm

In [ ]:
# =========================================================
# CELL 8: Train Seg+Cls (sequentially per dataset)
# =========================================================
for dataset_name in DATASET_NAMES:
    print(f"\n=== Seg+Cls training for {dataset_name} ===")
    train_segcls = SegClsDataset("train", dataset_name)
    val_segcls   = SegClsDataset("val", dataset_name)

    train_loader_segcls = DataLoader(train_segcls, batch_size=BATCH_SIZE, shuffle=True,
                                     num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS>0))
    val_loader_segcls = DataLoader(val_segcls, batch_size=BATCH_SIZE, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS>0))

    segcls_model = SegClsNet().to(DEVICE)
    opt_segcls = torch.optim.AdamW(segcls_model.parameters(), lr=1e-4, weight_decay=1e-4)
    sch_segcls = torch.optim.lr_scheduler.CosineAnnealingLR(opt_segcls, T_max=EPOCHS_SEGCLS)
    bce_cls = nn.BCEWithLogitsLoss()
    bce_seg = nn.BCEWithLogitsLoss()

    best_val = 1e9
    SEGCLS_CKPT = f"/content/best_segcls_{dataset_name}.pth"

    for ep in range(1, EPOCHS_SEGCLS + 1):
        tr = run_segcls_epoch(segcls_model, train_loader_segcls, opt_segcls, bce_cls, bce_seg, train=True)
        va = run_segcls_epoch(segcls_model, val_loader_segcls, None, bce_cls, bce_seg, train=False)
        sch_segcls.step()

        tr_loss, tr_cls, tr_seg, tr_acc, tr_p, tr_r, tr_f1, tr_cm = tr
        va_loss, va_cls, va_seg, va_acc, va_p, va_r, va_f1, va_cm = va

        print(f"\n[EP {ep}/{EPOCHS_SEGCLS}]"
              f"\n Train: loss={tr_loss:.4f} cls={tr_cls:.4f} seg={tr_seg:.4f} acc={tr_acc:.4f} f1={tr_f1:.4f}"
              f"\n Val  : loss={va_loss:.4f} cls={va_cls:.4f} seg={va_seg:.4f} acc={va_acc:.4f} f1={va_f1:.4f}")
        print("Val CM:\n", va_cm)

        if va_loss < best_val:
            best_val = va_loss
            torch.save(segcls_model.state_dict(), SEGCLS_CKPT)
            print("Saved:", SEGCLS_CKPT)

In [ ]:
# =========================================================
# CELL 9: Detection Dataset (Faster R-CNN)
# =========================================================
class CocoDetectionDataset(Dataset):
    def __init__(self, split, dataset_name):
        self.images, self.anns_by_img, self.lookup, self.img_dir, _ = load_coco(split, dataset_name)
        self.tfm = get_det_tfm(train=(split=="train"))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        item = self.images[idx]
        img_id = item["id"]

        img_path = self.img_dir / self.lookup[img_id]
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(str(img_path))
        img = np.stack([img, img, img], axis=-1)  # HWC RGB-like

        anns = self.anns_by_img.get(img_id, [])

        boxes = []
        labels = []
        for a in anns:
            if "bbox" in a and a["bbox"][2] > 1 and a["bbox"][3] > 1:
                boxes.append(coco_bbox_xywh_to_xyxy(a["bbox"]))
                labels.append(1)  # single object class

        # Albumentations bbox transform
        if len(boxes) == 0:
            # keep one dummy tiny box? no. Better allow empty targets.
            aug = self.tfm(image=img, bboxes=[], labels=[])
            img_aug = aug["image"]
            boxes_aug = []
            labels_aug = []
        else:
            aug = self.tfm(image=img, bboxes=boxes, labels=labels)
            img_aug = aug["image"]
            boxes_aug = aug["bboxes"]
            labels_aug = aug["labels"]

        # to tensor image [C,H,W], normalize [0,1]
        img_t = torch.tensor(img_aug).permute(2,0,1).float() / 255.0

        if len(boxes_aug) > 0:
            boxes_t = torch.tensor(boxes_aug, dtype=torch.float32)
            labels_t = torch.tensor(labels_aug, dtype=torch.int64)
            areas_t = (boxes_t[:,2]-boxes_t[:,0]) * (boxes_t[:,3]-boxes_t[:,1])
        else:
            boxes_t = torch.zeros((0,4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)
            areas_t = torch.zeros((0,), dtype=torch.float32)

        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "image_id": torch.tensor([img_id]),
            "area": areas_t,
            "iscrowd": torch.zeros((len(labels_t),), dtype=torch.int64)
        }

        return img_t, target

def det_collate_fn(batch):
    imgs, tgts = zip(*batch)
    return list(imgs), list(tgts)

In [ ]:
# =========================================================
# CELL 10: Detection model + train loop
# =========================================================
def get_frcnn_model(num_classes=2):
    # classes: background(0), lesion(1)
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
    return model

def train_det_one_epoch(model, loader, optimizer, epoch):
    model.train()
    total = 0.0
    for i, (images, targets) in enumerate(loader):
        images = [im.to(DEVICE) for im in images]
        targets = [{k:v.to(DEVICE) for k,v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

        if i % 25 == 0:
            ld = {k: float(v.detach().cpu()) for k,v in loss_dict.items()}
            print(f"[DET][EP{epoch}] batch {i}/{len(loader)} total={loss.item():.4f} {ld}", flush=True)

    return total / max(1, len(loader))

@torch.no_grad()
def eval_det_simple(model, loader, score_thr=0.3, iou_thr=0.5):
    """
    Simple proxy metric:
    - for each GT image, match predicted boxes with GT by IoU>=0.5
    - compute precision/recall over all predicted and GT boxes
    """
    model.eval()
    TP, FP, FN = 0, 0, 0

    for images, targets in loader:
        images = [im.to(DEVICE) for im in images]
        outputs = model(images)

        for out, tgt in zip(outputs, targets):
            gt_boxes = tgt["boxes"].numpy() if isinstance(tgt["boxes"], torch.Tensor) else tgt["boxes"]
            pred_boxes = out["boxes"].detach().cpu().numpy()
            pred_scores = out["scores"].detach().cpu().numpy()

            keep = pred_scores >= score_thr
            pred_boxes = pred_boxes[keep]

            if len(gt_boxes) == 0 and len(pred_boxes) == 0:
                continue
            if len(gt_boxes) == 0:
                FP += len(pred_boxes)
                continue
            if len(pred_boxes) == 0:
                FN += len(gt_boxes)
                continue

            ious = box_iou(torch.tensor(pred_boxes), torch.tensor(gt_boxes)).numpy()
            matched_gt = set()
            tp_here = 0

            for i in range(ious.shape[0]):
                j = np.argmax(ious[i])
                if ious[i, j] >= iou_thr and j not in matched_gt:
                    matched_gt.add(j)
                    tp_here += 1
                else:
                    FP += 1

            TP += tp_here
            FN += (len(gt_boxes) - tp_here)

    precision = TP / (TP + FP + 1e-9)
    recall = TP / (TP + FN + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return precision, recall, f1

In [ ]:
# =========================================================
# CELL 11: Train Detection (sequentially per dataset)
# =========================================================
for dataset_name in DATASET_NAMES:
    print(f"\n=== Detection training for {dataset_name} ===")
    train_det = CocoDetectionDataset("train", dataset_name)
    val_det   = CocoDetectionDataset("val", dataset_name)

    train_loader_det = DataLoader(train_det, batch_size=2, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True,
                                  collate_fn=det_collate_fn, persistent_workers=(NUM_WORKERS>0))
    val_loader_det = DataLoader(val_det, batch_size=2, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True,
                                collate_fn=det_collate_fn, persistent_workers=(NUM_WORKERS>0))

    det_model = get_frcnn_model(num_classes=2).to(DEVICE)
    opt_det = torch.optim.AdamW(det_model.parameters(), lr=1e-4, weight_decay=1e-4)

    DET_CKPT = f"/content/best_det_frcnn_{dataset_name}.pth"
    best_det_f1 = -1

    for ep in range(1, EPOCHS_DET + 1):
        tr_loss = train_det_one_epoch(det_model, train_loader_det, opt_det, ep)
        p, r, f1 = eval_det_simple(det_model, val_loader_det, score_thr=0.3, iou_thr=0.5)

        print(f"\n[DET EP {ep}/{EPOCHS_DET}] train_loss={tr_loss:.4f} val_precision={p:.4f} val_recall={r:.4f} val_f1={f1:.4f}")

        if f1 > best_det_f1:
            best_det_f1 = f1
            torch.save(det_model.state_dict(), DET_CKPT)
            print("Saved:", DET_CKPT)